In [25]:
import os
import json
from scraper import fetch_website_links, fetch_website_contents
from dotenv import load_dotenv
load_dotenv(override=True)

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



API key found and looks good so far!


In [26]:
from openai import OpenAI
ollama_base_url = "http://localhost:11434/v1"
ollama= OpenAI(base_url=ollama_base_url, api_key="ollama")
response = ollama.chat.completions.create(model="llama3.2", messages=[{
    "role": "user",
    "content": "Tell me a fun fact"
}])

response.choices[0].message.content

"Here's a fun fact:\n\nDid you know that honey never spoils? Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still perfectly edible! Honey's unique combination of acidity and water content, as well as its low moisture content, make it a virtually self-preserving food that can withstand extreme temperatures and conditions. Isn't that sweet?"

In [27]:

# The article to build the cheat sheet from - change this to target a different page
ARTICLE = "https://alistapart.com/article/css-positioning-101/"


In [28]:
# system prompt instructions to sort the relavent links from the list
system_prompt = """
You are provided with a list of links found in a website about css layout. As such you want to figure out which links might be useful for someone to use to stude more about css.
In this case, you only want to save links that are articles.
You should respond in JSON format as in this example.
{
    "links": [
        {"type": "css layout article", "url": "https://alistapart.com/article/css-positioning-101/"}
    ]
}
"""

def get_links_user_prompt(url):
    # user prompt to get the complete list of links from the website
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web link to include a cheat sheet about css layout.
    Respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.
    Limit the returned links to 10.
    Only save links that seem to be articles, the relavent links will have the word 'article' in the url in the breadcrumb. Do not save link that are about pages, or about the author, or about the website.
    Links (some might be relative links):

    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    # return the user prompt with all the links
    return user_prompt


In [32]:
def select_relavent_links(url):
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    data = json.loads(result)

    # Normalize links so each entry is always {"type": ..., "url": ...}.
    # llama3.2 sometimes returns bare URL strings instead of objects.
    normalized = []
    for link in data.get("links", []):
        if isinstance(link, str):
            normalized.append({"type": "css layout article", "url": link})
        elif isinstance(link, dict) and "url" in link:
            normalized.append({"type": link.get("type", "css layout article"), "url": link["url"]})
    return {"links": normalized}

In [33]:
selected_links = select_relavent_links(ARTICLE)
print(selected_links)

{'links': ['https://alistapart.com/articles/css-positioning-101/', 'https://alistapart.com/article/css-positioning-101/', 'https://alistapart.com/article/css-floats-101/', 'https://alistapart.com/article/mobile-first-css-is-it-time-for-a-rethink/', 'https://alistapart.com/article/breaking-out-of-the-box/', 'https://alistapart.com/article/designed-for-a-dead-language/', 'https://alistapart.com/article/design-for-amiability-lessons-from-vienna/', 'https://alistapart.com/article/design-dialects-breaking-the-rules-not-the-system/']}


In [34]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relavent_links(url)
    result = f"## CSS Layout Cheat Sheet:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result    

In [36]:
relative_page_date = fetch_page_and_all_relevant_links(ARTICLE)
print(relative_page_date)

TypeError: string indices must be integers, not 'str'